# 🧠 Stroke Detection via Facial Asymmetry
## Using MediaPipe Landmarks + Machine Learning

---

**Project:** Real-time stroke screening through facial asymmetry analysis  
**Method:** MediaPipe Face Mesh → Feature Extraction → ML Classifier  
**Output:** `stroke_model.pkl` and `stroke_scaler.pkl`  
**Version:** V_08 — Added SHAP explainability, improved camera FPS, XGBoost candidate, Section 4C pairplot, PDF output path fix

---

### Clinical Background

This project is grounded in the **FAST protocol** (Face, Arm, Speech, Time) —  
the internationally validated stroke screening tool. The **"F" (Face drooping)**  
component is the core of our detection system.

> *"Facial asymmetry is a key clinical indicator of stroke. However, human  
> assessment is highly subjective — paramedics failed to detect facial weakness  
> in 17% of stroke patients."*  
> — Herpich et al., Frontiers in Neurology (2022)

**References used in this project:**
1. Herpich et al. (2022) — Human vs. ML Detection of Facial Weakness, *Frontiers in Neurology*
https://www.frontiersin.org/journals/neurology/articles/10.3389/fneur.2022.878282/full
2. Taufique & Savakis (2021) — Automatic Quantification of Facial Asymmetry using Facial Landmarks, *arXiv:2103.11059*
https://arxiv.org/pdf/2103.11059
3. Oliveira et al. (2023) — Facial Point Graphs for Stroke Identification
https://www.researchgate.net/publication/375956719_Facial_Point_Graphs_for_Stroke_Identification
4. Frontera et al. (2026) — Systematic Review of FER in Stroke, *Frontiers in Neurology*
https://www.frontiersin.org/journals/neurology/articles/10.3389/fneur.2026.1744257/full
5. Baptista et al. (2022) — Systematic Review Comparing FAST and BE-FAST, *PMC8837419*
https://pmc.ncbi.nlm.nih.gov/articles/PMC8837419/
---

### Why Synthetic Data for Training?

The only publicly available stroke facial dataset is the **Toronto Neuroface (TNF)**  
dataset — which contains data from only **14 stroke patients** (Frontera et al., 2026).  
This is insufficient for training a reliable classifier.

Therefore, we generate a **clinically-inspired synthetic dataset** based on:
- FAST protocol thresholds (Kothari et al., 1999)
- Asymmetry score equations (Taufique & Savakis, 2021)
- Sunnybrook Facial Grading System criteria (Ross et al., 1996)

Real-world images from the **Annotated stroke and non stroke Dataset** are used  
**only for model selection** — to determine which trained model generalises best  
to actual clinical images (noise, lighting variation, angle).
https://www.kaggle.com/datasets/abdussalamelhanashy/annotated-facial-images-for-stroke-classification

---

### V_08 Improvements at a Glance
| # | Change | Benefit |
|---|--------|---------|
| 1 | **XGBoost** added as 4th model candidate | Better real-world generalisation |
| 2 | **SHAP** bar chart (Section 5C) | Explainable AI — see which feature drives the model |
| 3 | **Section 4C** Pairplot | Quick visual sanity-check of synthetic data separation |
| 4 | **Camera FPS boost** — preview resize + skip logic | Smoother inline streaming |
| 5 | **PDF path** uses `os.path.expanduser` (cross-platform) | Works on Windows, macOS, Linux |
| 6 | **Section 7** prints per-feature flag table | Better quick-validation readout |


## Section 0 — Environment Setup (RUN THIS FIRST)

**Known issue:** some recent `mediapipe` builds break `mp.solutions` on Python 3.12.  
The cell below auto-detects and fixes this.

> ⚠️ **If this cell installs anything, RESTART the kernel then run it again.**


In [ ]:
# ─────────────────────────────────────────────
# Environment check & auto-fix for mediapipe
# ─────────────────────────────────────────────
import subprocess, sys

def check_mediapipe():
    try:
        import mediapipe as mp
        return True, getattr(mp, '__version__', 'unknown'), hasattr(mp, 'solutions')
    except ImportError:
        return False, None, False

def pip_install(*packages):
    cmd = [sys.executable, "-m", "pip", "install", "--quiet", *packages]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0 and "externally-managed-environment" in result.stderr:
        cmd = [sys.executable, "-m", "pip", "install", "--quiet",
               "--break-system-packages", *packages]
        result = subprocess.run(cmd, capture_output=True, text=True)
    return result

installed, version, solutions_ok = check_mediapipe()
print(f"mediapipe installed : {installed}")
print(f"mediapipe version   : {version}")
print(f"mp.solutions OK     : {solutions_ok}")

if not installed or not solutions_ok:
    print("\n⚠️  Installing mediapipe==0.10.14 + numpy<2 ...")
    result = pip_install("mediapipe==0.10.14", "numpy<2")
    if result.returncode != 0:
        print("❌ Auto-install failed. Run manually:")
        print('   pip install "mediapipe==0.10.14" "numpy<2"')
        print(result.stderr[-800:])
    else:
        print("✅ Done.  🔁 RESTART THE KERNEL now, then re-run this cell.")
else:
    print("\n✅ mediapipe OK — ready to go!")


## Section 1 — Import Libraries

In [ ]:
import os, glob, warnings
import numpy as np
import pandas as pd
import joblib
import cv2
import mediapipe as mp
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, roc_curve, accuracy_score,
                              recall_score, precision_score, f1_score)

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

# ── Optional: XGBoost ────────────────────────────────────────
try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
    print("✅ XGBoost available — will be included as model candidate.")
except ImportError:
    XGBOOST_AVAILABLE = False
    print("ℹ️  XGBoost not installed — skipping. Install with: pip install xgboost")

# ── Optional: SHAP ───────────────────────────────────────────
try:
    import shap
    SHAP_AVAILABLE = True
    print("✅ SHAP available — explainability plots enabled.")
except ImportError:
    SHAP_AVAILABLE = False
    print("ℹ️  SHAP not installed — skipping. Install with: pip install shap")

sns.set_theme(style="darkgrid", palette="Set2")
plt.rcParams.update({'figure.dpi': 110, 'axes.titlesize': 13,
                     'axes.labelsize': 11, 'figure.facecolor': '#0f0f1a',
                     'axes.facecolor': '#1a1a2e', 'axes.edgecolor': '#444',
                     'text.color': 'white', 'axes.labelcolor': 'white',
                     'xtick.color': 'white', 'ytick.color': 'white',
                     'grid.color': '#333355', 'grid.linestyle': '--'})

# ── Resolve project root (no hardcoded paths) ──────────────────
try:
    NOTEBOOK_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    NOTEBOOK_DIR = os.getcwd()

DATASET_DIR    = os.path.join(NOTEBOOK_DIR, 'Annotated stroke and non stroke Dataset')
STROKE_DIR     = os.path.join(DATASET_DIR, 'Stroke')
NON_STROKE_DIR = os.path.join(DATASET_DIR, 'NonStroke')
CACHE_CSV      = os.path.join(NOTEBOOK_DIR, 'real_world_validation_features.csv')
MODEL_PATH     = os.path.join(NOTEBOOK_DIR, 'stroke_model.pkl')
SCALER_PATH    = os.path.join(NOTEBOOK_DIR, 'stroke_scaler.pkl')

print("✅ Libraries loaded.")
print(f"   Project root   : {NOTEBOOK_DIR}")
print(f"   Dataset folder : {DATASET_DIR}")
print(f"   Exists?        : {os.path.exists(DATASET_DIR)}")


## Section 2 — MediaPipe Landmark Indices & Feature Functions

| Feature | Clinical Basis | Landmarks Used |
|---|---|---|
| `eye_asymmetry` | Eye drooping (ptosis) — FAST "F" | Top/bottom eyelid L/R |
| `mouth_corner_drop` | Mouth droop — FAST "F" | Mouth corners L/R |
| `eyebrow_height_diff` | Brow paralysis — Sunnybrook | Eyebrow center L/R |
| `nasolabial_asymmetry` | Nasolabial fold — stroke sign | Nose-to-fold L/R |
| `face_midline_deviation` | Overall face shift | Nose vs face centre |
| `mouth_width_asymmetry` | Mouth width ratio — CPSS | Width left/right half |


In [ ]:
# ─────────────────────────────────────────────
# Landmark indices (MediaPipe 468-point model)
# ─────────────────────────────────────────────
LANDMARKS = {
    'left_eye_top': 159,  'left_eye_bottom': 145,
    'right_eye_top': 386, 'right_eye_bottom': 374,
    'mouth_left': 61,     'mouth_right': 291,
    'mouth_top': 13,      'mouth_bottom': 14,
    'left_eyebrow': 70,   'right_eyebrow': 300,
    'nose_tip': 1,
    'left_nasolabial': 92, 'right_nasolabial': 322,
    'left_face_edge': 234, 'right_face_edge': 454,
    'chin': 152,           'forehead': 10,
}

FEATURE_COLS = [
    'eye_asymmetry', 'mouth_corner_drop', 'eyebrow_height_diff',
    'nasolabial_asymmetry', 'face_midline_deviation', 'mouth_width_asymmetry'
]

THRESHOLDS = {
    'eye_asymmetry': 0.12, 'mouth_corner_drop': 0.10,
    'eyebrow_height_diff': 0.08, 'nasolabial_asymmetry': 0.10,
    'face_midline_deviation': 0.05, 'mouth_width_asymmetry': 0.10,
}

def euclidean(p1, p2):
    return np.sqrt((p1.x - p2.x)**2 + (p1.y - p2.y)**2)

def extract_features(landmarks):
    lm = landmarks
    left_eye_h  = abs(lm[159].y - lm[145].y)
    right_eye_h = abs(lm[386].y - lm[374].y)
    eye_asym = abs(left_eye_h - right_eye_h) / (max(left_eye_h, right_eye_h) + 1e-6)

    face_height = abs(lm[10].y - lm[152].y) + 1e-6
    mouth_drop  = abs(lm[61].y - lm[291].y) / face_height

    left_brow  = abs(lm[70].y  - lm[159].y)
    right_brow = abs(lm[300].y - lm[386].y)
    brow_diff  = abs(left_brow - right_brow) / (max(left_brow, right_brow) + 1e-6)

    n2l = euclidean(lm[1], lm[92])
    n2r = euclidean(lm[1], lm[322])
    naso = abs(n2l - n2r) / (max(n2l, n2r) + 1e-6)

    face_cx  = (lm[234].x + lm[454].x) / 2
    face_w   = abs(lm[234].x - lm[454].x) + 1e-6
    midline  = abs(lm[1].x - face_cx) / face_w

    mcx    = (lm[61].x + lm[291].x) / 2
    mw_l   = abs(lm[61].x  - mcx)
    mw_r   = abs(lm[291].x - mcx)
    mw_asy = abs(mw_l - mw_r) / (max(mw_l, mw_r) + 1e-6)

    return {
        'eye_asymmetry':          float(eye_asym),
        'mouth_corner_drop':      float(mouth_drop),
        'eyebrow_height_diff':    float(brow_diff),
        'nasolabial_asymmetry':   float(naso),
        'face_midline_deviation': float(midline),
        'mouth_width_asymmetry':  float(mw_asy),
    }

def predict_stroke_risk(features_dict, model, scaler):
    X = np.array([[features_dict[f] for f in FEATURE_COLS]])
    X_sc = scaler.transform(X)
    prob  = model.predict_proba(X_sc)[0][1]
    label = model.predict(X_sc)[0]
    if prob < 0.25:
        risk, hint = "LOW RISK", "green"
    elif prob < 0.55:
        risk, hint = "MODERATE RISK — Consult a doctor", "orange"
    else:
        risk, hint = "HIGH RISK — Seek immediate medical help", "red"
    return {'probability': round(prob,4), 'percentage': round(prob*100,2),
            'label': int(label), 'risk_level': risk, 'color_hint': hint}

print("✅ Landmark indices & feature functions defined.")


## Section 3 — Synthetic Dataset Generation

Each feature's normal/stroke range is derived from clinical literature.  
A bimodal stroke distribution (60% mild + 40% severe) adds realistic variety.


In [ ]:
np.random.seed(42)
N_NORMAL = N_STROKE = 1500

def generate_normal_samples(n):
    return {
        'eye_asymmetry':          np.random.normal(0.04, 0.02, n).clip(0.0, 0.12),
        'mouth_corner_drop':      np.random.normal(0.03, 0.02, n).clip(0.0, 0.10),
        'eyebrow_height_diff':    np.random.normal(0.03, 0.015, n).clip(0.0, 0.08),
        'nasolabial_asymmetry':   np.random.normal(0.03, 0.02, n).clip(0.0, 0.10),
        'face_midline_deviation': np.random.normal(0.015, 0.01, n).clip(0.0, 0.05),
        'mouth_width_asymmetry':  np.random.normal(0.04, 0.02, n).clip(0.0, 0.10),
    }

def generate_stroke_samples(n):
    mild, severe = int(n * 0.6), n - int(n * 0.6)
    def cb(a, b): return np.concatenate([a, b])
    return {
        'eye_asymmetry':          cb(np.random.normal(0.18,0.04,mild).clip(0.12,0.30),
                                      np.random.normal(0.35,0.06,severe).clip(0.25,0.55)),
        'mouth_corner_drop':      cb(np.random.normal(0.20,0.05,mild).clip(0.12,0.35),
                                      np.random.normal(0.40,0.07,severe).clip(0.30,0.60)),
        'eyebrow_height_diff':    cb(np.random.normal(0.15,0.04,mild).clip(0.08,0.25),
                                      np.random.normal(0.30,0.06,severe).clip(0.20,0.50)),
        'nasolabial_asymmetry':   cb(np.random.normal(0.18,0.05,mild).clip(0.10,0.30),
                                      np.random.normal(0.35,0.07,severe).clip(0.25,0.55)),
        'face_midline_deviation': cb(np.random.normal(0.06,0.02,mild).clip(0.03,0.12),
                                      np.random.normal(0.12,0.03,severe).clip(0.08,0.22)),
        'mouth_width_asymmetry':  cb(np.random.normal(0.18,0.04,mild).clip(0.10,0.28),
                                      np.random.normal(0.32,0.06,severe).clip(0.22,0.50)),
    }

normal_df = pd.DataFrame(generate_normal_samples(N_NORMAL)); normal_df['label'] = 0
stroke_df = pd.DataFrame(generate_stroke_samples(N_STROKE)); stroke_df['label'] = 1
df = pd.concat([normal_df, stroke_df], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)

# ── Auto-save synthetic dataset to CSV ──────────────────────
SYNTHETIC_CSV = os.path.join(NOTEBOOK_DIR, 'synthetic_dataset.csv')
df.to_csv(SYNTHETIC_CSV, index=False)

print(f"✅ Synthetic dataset: {len(df)} samples  |  Normal: {(df.label==0).sum()}  |  Stroke: {(df.label==1).sum()}")
print(f"✅ Saved → {SYNTHETIC_CSV}")
df.head()


## Section 4 — Exploratory Data Analysis (EDA)

In [ ]:
# ── Distribution plots ────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(17, 9))
fig.patch.set_facecolor('#0f0f1a')
axes = axes.flatten()

PALETTE = {0: ('#2ecc71', 'Normal'), 1: ('#e74c3c', 'Stroke Risk')}

for i, feat in enumerate(FEATURE_COLS):
    ax = axes[i]
    for lv, (col, lbl) in PALETTE.items():
        data = df[df.label == lv][feat]
        ax.hist(data, bins=45, alpha=0.65, color=col, label=lbl, density=True)
        # KDE overlay
        from scipy.stats import gaussian_kde
        xs = np.linspace(data.min(), data.max(), 300)
        ax.plot(xs, gaussian_kde(data)(xs), color=col, lw=2)
    thresh = THRESHOLDS[feat]
    ax.axvline(thresh, color='#f1c40f', lw=1.5, linestyle='--', label=f'Threshold ({thresh})')
    ax.set_title(feat.replace('_',' ').title(), fontweight='bold')
    ax.set_xlabel('Asymmetry Score')
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)

fig.suptitle('Feature Distributions — Normal vs Stroke Risk\n'
             '(Yellow dashed = clinical threshold | KDE overlay)',
             fontsize=14, fontweight='bold', color='white', y=1.01)
plt.tight_layout()
plt.savefig('eda_distributions.png', dpi=130, bbox_inches='tight', facecolor='#0f0f1a')
plt.show()
print("✅ Distribution plot saved → eda_distributions.png")


In [ ]:
# ── Correlation heatmap ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))
fig.patch.set_facecolor('#0f0f1a')
ax.set_facecolor('#1a1a2e')
corr = df[FEATURE_COLS].corr()
mask = np.tril(np.ones_like(corr, dtype=bool))   # show upper triangle

sns.heatmap(corr, mask=~mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=ax, linewidths=0.6, annot_kws={'size': 10},
            cbar_kws={'shrink': 0.8})

ax.set_title('Feature Correlation Matrix\n(Low = each feature carries unique info)',
             fontweight='bold', color='white', pad=12)
plt.tight_layout()
plt.savefig('eda_correlation.png', dpi=130, bbox_inches='tight', facecolor='#0f0f1a')
plt.show()
print("✅ Correlation heatmap saved → eda_correlation.png")


## Section 4B — Real-World Feature Extraction

MediaPipe extracts the 6 asymmetry features from each image in  
`Annotated stroke and non stroke Dataset/Stroke/` (label=1) and  
`Annotated stroke and non stroke Dataset/NonStroke/` (label=0).

**100 images per class** are sampled randomly for balanced evaluation.  
Results are cached to `real_world_validation_features.csv` so  
subsequent runs skip the slow extraction step.


In [ ]:
import random

N_PER_CLASS = 100   # images per class used for validation
IMG_EXTS    = ('*.jpg','*.jpeg','*.png','*.webp','*.bmp')

def collect_images(folder):
    imgs = []
    for ext in IMG_EXTS:
        imgs.extend(glob.glob(os.path.join(folder, ext)))
    return sorted(imgs)

if os.path.exists(CACHE_CSV):
    val_df = pd.read_csv(CACHE_CSV)
    print(f"✅ Loaded cached validation features ({len(val_df)} samples) from\n   {CACHE_CSV}")
else:
    for d, name in [(STROKE_DIR,'Stroke'), (NON_STROKE_DIR,'NonStroke')]:
        if not os.path.exists(d):
            raise FileNotFoundError(
                f"❌ '{name}' folder not found at:\n   {d}\n"
                "   Place 'Annotated stroke and non stroke Dataset' next to this notebook.")

    stroke_imgs     = collect_images(STROKE_DIR)
    non_stroke_imgs = collect_images(NON_STROKE_DIR)

    print(f"Found {len(stroke_imgs):>4} Stroke images")
    print(f"Found {len(non_stroke_imgs):>4} NonStroke images")

    # ── Sample N_PER_CLASS from each ──
    rng = random.Random(42)
    stroke_sample     = rng.sample(stroke_imgs,     min(N_PER_CLASS, len(stroke_imgs)))
    non_stroke_sample = rng.sample(non_stroke_imgs, min(N_PER_CLASS, len(non_stroke_imgs)))

    all_tasks = [(p,1) for p in stroke_sample] + [(p,0) for p in non_stroke_sample]
    print(f"\nUsing {len(stroke_sample)} Stroke + {len(non_stroke_sample)} NonStroke images for validation.")
    print("Extracting features with MediaPipe ...\n")

    mp_fm = mp.solutions.face_mesh
    face_mesh = mp_fm.FaceMesh(static_image_mode=True, max_num_faces=1,
                                refine_landmarks=True, min_detection_confidence=0.5)
    records, skipped = [], 0

    for idx, (img_path, label) in enumerate(all_tasks):
        prog = f"  [{idx+1:>3}/{len(all_tasks)}]"
        img = cv2.imread(img_path)
        if img is None:
            skipped += 1; continue
        results = face_mesh.process(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        if not results.multi_face_landmarks:
            skipped += 1; continue
        try:
            feats = extract_features(results.multi_face_landmarks[0].landmark)
            feats['label'] = label
            feats['source'] = 'stroke' if label == 1 else 'non_stroke'
            records.append(feats)
        except Exception:
            skipped += 1

        if (idx+1) % 50 == 0 or (idx+1) == len(all_tasks):
            print(f"{prog}  processed={len(records)}  skipped={skipped}")

    face_mesh.close()
    val_df = pd.DataFrame(records)
    val_df.to_csv(CACHE_CSV, index=False)

    print(f"\n✅ Extraction complete!")
    print(f"   Usable samples : {len(val_df)}  |  Skipped: {skipped}")
    print(f"   Stroke (1)     : {(val_df.label==1).sum()}")
    print(f"   NonStroke (0)  : {(val_df.label==0).sum()}")
    print(f"   Saved → {CACHE_CSV}")

val_df.head()


## Section 4C — Pairplot (V_08 New)

Quick visual sanity-check: are the 6 features actually separating normal vs stroke in 2D projections?
Diagonal = KDE of each feature. Off-diagonal = scatter of feature pairs.
> Run only once — takes ~10s on 3000 samples.


In [ ]:
# ── Section 4C: Pairplot (V_08) ─────────────────────────────────────
# Subsample for speed: 300 points per class
PP_N = 300
pp_df = pd.concat([
    df[df.label == 0].sample(PP_N, random_state=42),
    df[df.label == 1].sample(PP_N, random_state=42),
]).reset_index(drop=True)
pp_df['Class'] = pp_df['label'].map({0: 'Normal', 1: 'Stroke Risk'})

with plt.rc_context({
    'figure.facecolor': '#0f0f1a', 'axes.facecolor': '#1a1a2e',
    'text.color': 'white', 'axes.labelcolor': 'white',
    'xtick.color': 'white', 'ytick.color': 'white',
    'axes.edgecolor': '#444', 'grid.color': '#333355'
}):
    pair_grid = sns.pairplot(
        pp_df, vars=FEATURE_COLS, hue='Class',
        palette={'Normal': '#2ecc71', 'Stroke Risk': '#e74c3c'},
        plot_kws={'alpha': 0.35, 's': 12},
        diag_kind='kde', corner=True
    )
    pair_grid.figure.suptitle(
        'Feature Pairplot — Normal vs Stroke Risk (300 samples/class)',
        y=1.01, fontsize=13, fontweight='bold', color='white'
    )
    # Style each axis
    for ax in pair_grid.axes.flatten():
        if ax is not None:
            ax.set_facecolor('#1a1a2e')
            ax.tick_params(colors='white')
    pair_grid.figure.patch.set_facecolor('#0f0f1a')
    plt.tight_layout()
    plt.savefig('eda_pairplot.png', dpi=110, bbox_inches='tight', facecolor='#0f0f1a')
    plt.show()

print('✅ Pairplot saved → eda_pairplot.png')


## Section 5 — Model Training & Real-World Validation

**Training data:** synthetic dataset (3 000 samples)  
**Model selection:** based on performance on 200 real-world images  
(100 Stroke + 100 NonStroke from the annotated dataset)

Priority metric: **Recall** (minimise missed strokes) then **AUC-ROC**.


In [ ]:
X_synth  = df[FEATURE_COLS].values
y_synth  = df['label'].values
X_train, X_test_synth, y_train, y_test_synth = train_test_split(
    X_synth, y_synth, test_size=0.2, random_state=42, stratify=y_synth)

X_real = val_df[FEATURE_COLS].values
y_real = val_df['label'].values

scaler = StandardScaler()
X_train_sc      = scaler.fit_transform(X_train)
X_test_synth_sc = scaler.transform(X_test_synth)
X_real_sc       = scaler.transform(X_real)

MODELS = {
    'Logistic Regression': LogisticRegression(max_iter=1000, C=1.0, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=300, max_depth=8,
                                                   min_samples_leaf=3, random_state=42, n_jobs=-1),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=300, max_depth=4,
                                                       learning_rate=0.04, subsample=0.8,
                                                       random_state=42),
}

# ── Add XGBoost if available ──────────────────────────────────
if XGBOOST_AVAILABLE:
    MODELS['XGBoost'] = XGBClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.04,
        subsample=0.8, colsample_bytree=0.8,
        use_label_encoder=False, eval_metric='logloss',
        random_state=42, n_jobs=-1
    )

kfold   = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}
W = 96

print("=" * W)
print(f"{'Model':<22} | {'CV Acc':>7} | {'±':>6} | {'Synth Acc':>9} | {'Real Acc':>8} | {'Recall':>7} | {'AUC':>7}")
print("=" * W)

for name, model in MODELS.items():
    cv   = cross_val_score(model, X_train_sc, y_train, cv=kfold, scoring='accuracy')
    model.fit(X_train_sc, y_train)

    y_ps = model.predict(X_test_synth_sc)
    y_pr = model.predict(X_real_sc)
    y_pb = model.predict_proba(X_real_sc)[:, 1]

    results[name] = {
        'model': model,
        'cv_mean': cv.mean(), 'cv_std': cv.std(),
        'synth_acc': accuracy_score(y_test_synth, y_ps),
        'real_acc':  accuracy_score(y_real, y_pr),
        'real_recall':    recall_score(y_real, y_pr, zero_division=0),
        'real_precision': precision_score(y_real, y_pr, zero_division=0),
        'real_f1':  f1_score(y_real, y_pr, zero_division=0),
        'real_auc': roc_auc_score(y_real, y_pb),
        'y_pred': y_pr, 'y_prob': y_pb,
    }
    r = results[name]
    print(f"{name:<22} | {r['cv_mean']:.4f} | {r['cv_std']:.4f} | {r['synth_acc']:.4f}    | "
          f"{r['real_acc']:.4f}   | {r['real_recall']:.4f}  | {r['real_auc']:.4f}")

print("=" * W)


In [ ]:
# ── Select best model ────────────────────────────────────────
best_name = max(results, key=lambda k: (results[k]['real_recall'], results[k]['real_auc']))
best = results[best_name]

print(f"\n🏆 Best Model: {best_name}")
print(f"   Synthetic Test Accuracy : {best['synth_acc']:.4f}")
print(f"   Real-World Accuracy     : {best['real_acc']:.4f}  ({best['real_acc']*100:.1f}%)")
print(f"   Real-World Recall       : {best['real_recall']:.4f} ({best['real_recall']*100:.1f}%)")
print(f"   Real-World AUC          : {best['real_auc']:.4f}")
print(f"\n📋 Classification Report:")
print(classification_report(y_real, best['y_pred'], target_names=['NonStroke','Stroke']))


In [ ]:
# ── Confusion Matrix + ROC + Model Comparison ─────────────────
fig = plt.figure(figsize=(18, 6), facecolor='#0f0f1a')
gs  = fig.add_gridspec(1, 3, wspace=0.38)

# 1) Confusion matrix
ax1 = fig.add_subplot(gs[0])
cm  = confusion_matrix(y_real, best['y_pred'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax1,
            xticklabels=['NonStroke','Stroke'], yticklabels=['NonStroke','Stroke'],
            linewidths=1.5, linecolor='#0f0f1a', annot_kws={'size':14,'weight':'bold'})
labels_map = {(0,0):'TN',(0,1):'FP',(1,0):'FN',(1,1):'TP'}
for (r,c), lbl in labels_map.items():
    ax1.text(c+0.5, r+0.78, f'({lbl})', ha='center', va='center', fontsize=9, color='#aaa')
ax1.set_title(f'Confusion Matrix\n{best_name}', fontweight='bold')
ax1.set_ylabel('Actual'); ax1.set_xlabel('Predicted')

# 2) ROC curves
ax2 = fig.add_subplot(gs[1])
MODEL_COLORS = {'Logistic Regression':'#3498db','Random Forest':'#2ecc71','Gradient Boosting':'#e74c3c'}
for n in MODELS:
    fpr, tpr, _ = roc_curve(y_real, results[n]['y_prob'])
    lw = 3 if n == best_name else 1.5
    ax2.plot(fpr, tpr, color=MODEL_COLORS[n], lw=lw,
             label=f'{n}\nAUC={results[n]["real_auc"]:.3f}')
ax2.plot([0,1],[0,1],'w--',lw=1,alpha=0.4,label='Random')
ax2.set_xlabel('False Positive Rate'); ax2.set_ylabel('True Positive Rate')
ax2.set_title('ROC Curves — Real-World Data', fontweight='bold')
ax2.legend(loc='lower right', fontsize=8, facecolor='#1a1a2e', edgecolor='#444', labelcolor='white')
ax2.set_xlim([0,1]); ax2.set_ylim([0,1.02])

# 3) Model comparison bar chart — Best model highlighted with crown
ax3 = fig.add_subplot(gs[2])
metrics = ['real_acc','real_recall','real_f1','real_auc']
metric_labels = ['Accuracy','Recall','F1','AUC']
x = np.arange(len(metrics))
width = 0.25
bar_groups = {}
for i, (n, col) in enumerate(MODEL_COLORS.items()):
    vals = [results[n][m] for m in metrics]
    # Best model gets full brightness, others get dimmed alpha
    alpha = 1.0 if n == best_name else 0.45
    edge  = '#FFD700' if n == best_name else '#0f0f1a'
    lw_b  = 2.0        if n == best_name else 0.5
    bars = ax3.bar(x + i*width, vals, width, label=n, color=col, alpha=alpha,
                   edgecolor=edge, linewidth=lw_b)
    bar_groups[n] = bars
    for b, v in zip(bars, vals):
        ax3.text(b.get_x()+b.get_width()/2, b.get_height()+0.008,
                 f'{v:.2f}', ha='center', va='bottom', fontsize=7, color='white')

# Crown + "BEST" label above the best model's first bar
best_bars = bar_groups[best_name]
bx = best_bars[0].get_x() + best_bars[0].get_width()/2
by = max(results[best_name][m] for m in metrics) + 0.08
ax3.text(bx + width, by + 0.015, f'👑 BEST\n{best_name}',
         ha='center', va='bottom', fontsize=8.5, color='#FFD700',
         fontweight='bold', linespacing=1.4,
         bbox=dict(boxstyle='round,pad=0.35', facecolor='#1a1a2e',
                   edgecolor='#FFD700', linewidth=1.5, alpha=0.9))

ax3.set_xticks(x + width); ax3.set_xticklabels(metric_labels)
ax3.set_ylim(0, 1.28)
ax3.set_title('Model Comparison — Real-World Metrics', fontweight='bold')
ax3.legend(fontsize=8, facecolor='#1a1a2e', edgecolor='#444', labelcolor='white')
ax3.set_ylabel('Score')

fig.suptitle(f'Model Evaluation on Real-World Annotated Dataset  |  Best: {best_name}',
             fontsize=14, fontweight='bold', color='white', y=1.02)
plt.savefig('model_evaluation.png', dpi=130, bbox_inches='tight', facecolor='#0f0f1a')
plt.show()
print("✅ Evaluation plots saved → model_evaluation.png")


In [ ]:
# ── Feature Importance ────────────────────────────────────────
model_obj = best['model']
fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor('#0f0f1a')
ax.set_facecolor('#1a1a2e')

if hasattr(model_obj, 'feature_importances_'):
    imp = model_obj.feature_importances_
    label_txt = 'Importance Score'
else:
    imp = np.abs(model_obj.coef_[0])
    label_txt = '|Coefficient|'

order  = np.argsort(imp)
colors = ['#e74c3c' if imp[i] == imp.max() else '#3498db' for i in order]
bars   = ax.barh([FEATURE_COLS[i].replace('_',' ').title() for i in order],
                  imp[order], color=colors, edgecolor='#0f0f1a', height=0.6)
for bar, val in zip(bars, imp[order]):
    ax.text(val + 0.003, bar.get_y()+bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=10, color='white', fontweight='bold')

ax.set_xlabel(label_txt)
ax.set_title(f'Feature Importance — {best_name}\n(Red bar = most discriminative feature)',
             fontweight='bold')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=130, bbox_inches='tight', facecolor='#0f0f1a')
plt.show()
print("✅ Feature importance saved → feature_importance.png")


## Section 5B — Save Champion Model

In [ ]:
joblib.dump(best['model'], MODEL_PATH)
joblib.dump(scaler,        SCALER_PATH)
print(f"✅ Model  saved → {MODEL_PATH}")
print(f"✅ Scaler saved → {SCALER_PATH}")
print(f"\nChampion : {best_name}")
print(f"Features : {FEATURE_COLS}")
print(f"Classes  : 0=NonStroke  1=Stroke")

# ── Quick reload test ──
_m = joblib.load(MODEL_PATH)
_s = joblib.load(SCALER_PATH)
print("\n✅ Reload test passed.")


## Section 5C — SHAP Feature Importance (V_08 New)

SHAP (SHapley Additive exPlanations) shows **which feature pushes the prediction** toward Stroke or Normal,  
and by how much — for both the champion model and globally across all test samples.

> Requires `shap` package: `pip install shap`  
> If unavailable, this section prints a fallback sklearn importance bar chart.


In [ ]:
# ── Section 5C: SHAP Explainability (V_08) ──────────────────────────────────────
import matplotlib.pyplot as plt

_model_obj = best['model']
_X_explain = X_real_sc   # explain on the real-world validation set

if SHAP_AVAILABLE:
    print("⚙️  Computing SHAP values (TreeExplainer)...")
    try:
        explainer   = shap.TreeExplainer(_model_obj)
        shap_values = explainer.shap_values(_X_explain)

        # For binary classification, shap_values may be [neg_class, pos_class]
        if isinstance(shap_values, list):
            sv = shap_values[1]   # class 1 = Stroke
        else:
            sv = shap_values

        mean_abs_shap = np.abs(sv).mean(axis=0)
        shap_df = pd.DataFrame({
            'Feature': FEATURE_COLS,
            'Mean |SHAP|': mean_abs_shap
        }).sort_values('Mean |SHAP|', ascending=True)

        fig, ax = plt.subplots(figsize=(9, 5))
        fig.patch.set_facecolor('#0f0f1a')
        ax.set_facecolor('#1a1a2e')

        colors_bar = ['#e74c3c' if v == shap_df['Mean |SHAP|'].max()
                      else '#3498db' for v in shap_df['Mean |SHAP|']]
        bars = ax.barh(shap_df['Feature'], shap_df['Mean |SHAP|'],
                       color=colors_bar, edgecolor='#444', linewidth=0.6)

        for bar, val in zip(bars, shap_df['Mean |SHAP|']):
            ax.text(val + 0.001, bar.get_y() + bar.get_height()/2,
                    f'{val:.4f}', va='center', ha='left',
                    color='white', fontsize=9)

        ax.set_xlabel('Mean |SHAP value| (impact on stroke prediction)')
        ax.set_title(f'SHAP Feature Importance — {best_name}\n'
                     f'(Red bar = strongest driver | higher = more influential)',
                     fontweight='bold', color='white')
        ax.tick_params(colors='white')
        ax.spines['bottom'].set_color('#444')
        ax.spines['left'].set_color('#444')
        plt.tight_layout()
        plt.savefig('shap_importance.png', dpi=130, bbox_inches='tight', facecolor='#0f0f1a')
        plt.show()
        print('✅ SHAP plot saved → shap_importance.png')

    except Exception as e:
        print(f'⚠️  TreeExplainer failed ({e}). Falling back to sklearn importance...')
        SHAP_AVAILABLE = False   # fall through

if not SHAP_AVAILABLE:
    # ── Fallback: sklearn feature importances ──
    print('ℹ️  Showing sklearn feature_importances_ (install shap for full SHAP).')
    if hasattr(_model_obj, 'feature_importances_'):
        fi = _model_obj.feature_importances_
    else:
        fi = np.abs(np.array([_model_obj.coef_[0]]))[0]

    fi_df = pd.DataFrame({'Feature': FEATURE_COLS, 'Importance': fi})\
              .sort_values('Importance', ascending=True)

    fig, ax = plt.subplots(figsize=(9, 5))
    fig.patch.set_facecolor('#0f0f1a'); ax.set_facecolor('#1a1a2e')
    ax.barh(fi_df['Feature'], fi_df['Importance'], color='#3498db', edgecolor='#444')
    ax.set_xlabel('Feature Importance')
    ax.set_title(f'Feature Importance — {best_name}\n(SHAP not available)',
                 fontweight='bold', color='white')
    ax.tick_params(colors='white')
    plt.tight_layout()
    plt.show()


## Section 6 — Real-Time Camera Detection (OpenCV)

Controls:
- **Q** → Quit
- **S** → Save screenshot

`FORCE_INLINE = True` streams inside the notebook cell (recommended).  
Set to `False` to open a native OpenCV window.


In [ ]:
import time
from IPython.display import clear_output

FORCE_INLINE  = True   # ← change to False for native window
MAX_FRAMES    = 300
PROCESS_EVERY = 2      # ← process every Nth frame (2=skip every other; increase to 3 if still slow)

# ── Load model ──
model  = joblib.load(MODEL_PATH)
scaler = joblib.load(SCALER_PATH)

# ── MediaPipe ──
mp_fm      = mp.solutions.face_mesh
mp_drawing = mp.solutions.drawing_utils
mp_styles  = mp.solutions.drawing_styles
face_mesh  = mp_fm.FaceMesh(max_num_faces=1, refine_landmarks=True,
                              min_detection_confidence=0.5, min_tracking_confidence=0.5)

# ── BGR color palette ──
C = {'green':(50,205,50),'orange':(0,165,255),'red':(50,50,220),
     'white':(255,255,255),'black':(0,0,0),'gray':(180,180,180),'cyan':(255,220,0)}

def draw_overlay(frame, features, pred, fps):
    h, w = frame.shape[:2]
    rc = C[pred['color_hint']]

    # Border
    cv2.rectangle(frame,(0,0),(w-1,h-1),rc,5)

    # Top bar
    cv2.rectangle(frame,(0,0),(w,48),(15,15,15),-1)
    cv2.putText(frame,'STROKE SCREENING',(10,32),cv2.FONT_HERSHEY_DUPLEX,0.8,C['cyan'],1)
    cv2.putText(frame,f'FPS:{fps:.0f}',(w-90,32),cv2.FONT_HERSHEY_SIMPLEX,0.55,C['gray'],1)

    # Feature panel
    px,py = 10,60
    cv2.rectangle(frame,(px-4,py-4),(px+240,py+160),(20,20,20),-1)
    feat_disp = {
        'eye_asymmetry':'Eye Asym','mouth_corner_drop':'Mouth Drop',
        'eyebrow_height_diff':'Brow Diff','nasolabial_asymmetry':'Nasolabial',
        'face_midline_deviation':'Midline','mouth_width_asymmetry':'Mouth Width',
    }
    for i,(k,lbl) in enumerate(feat_disp.items()):
        v = features[k]
        col = C['red'] if v > THRESHOLDS[k] else C['green']
        flag = '▲' if v > THRESHOLDS[k] else '✓'
        cv2.putText(frame,f'{flag} {lbl:<14}: {v:.3f}',(px,py+i*25+16),
                    cv2.FONT_HERSHEY_SIMPLEX,0.44,col,1)

    # Bottom risk banner
    bh = 78
    cv2.rectangle(frame,(0,h-bh),(w,h),(12,12,12),-1)
    cv2.putText(frame,f"Stroke Risk: {pred['percentage']:.1f}%",(10,h-bh+28),
                cv2.FONT_HERSHEY_DUPLEX,0.85,C['white'],1)
    cv2.putText(frame,pred['risk_level'],(10,h-bh+60),
                cv2.FONT_HERSHEY_SIMPLEX,0.6,rc,2)

    # Progress bar
    bx,by,bw = 10,h-10,w-20
    cv2.rectangle(frame,(bx,by-6),(bx+bw,by+6),(55,55,55),-1)
    fw = int(bw*pred['probability'])
    cv2.rectangle(frame,(bx,by-6),(bx+fw,by+6),rc,-1)
    return frame

def gui_ok():
    if FORCE_INLINE: return False
    try:
        cv2.namedWindow('__t__',cv2.WINDOW_NORMAL); cv2.destroyWindow('__t__'); return True
    except: return False

GUI = gui_ok()
print("🖥️  Native window" if GUI else "📺  Inline preview (interrupt cell to stop)")
print(f"⚡ Frame-skip mode: processing every {PROCESS_EVERY} frames (PROCESS_EVERY={PROCESS_EVERY})")

cap = cv2.VideoCapture(0)
# ── Lower resolution for speed; increase if image quality matters ──
cap.set(cv2.CAP_PROP_FRAME_WIDTH,  640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
cap.set(cv2.CAP_PROP_BUFFERSIZE,   1)   # prevent buffer lag

if not cap.isOpened():
    print("❌ Camera not found. Try VideoCapture(1) or run Section 8 for static image mode.")
    face_mesh.close()
else:
    print("🎥 Camera started.")
    prev, shots, frames   = time.time(), 0, 0
    frame_count           = 0
    last_feat, last_pred  = None, None   # cache last result for skipped frames

    try:
        while True:
            ret, frame = cap.read()
            if not ret: break
            frame = cv2.flip(frame, 1)
            frame_count += 1

            # ── Only run MediaPipe on every Nth frame ──
            if frame_count % PROCESS_EVERY == 0:
                rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                res = face_mesh.process(rgb)
                now = time.time(); fps = 1/(now-prev+1e-6); prev = now

                if res.multi_face_landmarks:
                    fl = res.multi_face_landmarks[0]
                    mp_drawing.draw_landmarks(frame, fl, mp_fm.FACEMESH_CONTOURS,
                        landmark_drawing_spec=None,
                        connection_drawing_spec=mp_styles.get_default_face_mesh_contours_style())
                    last_feat = extract_features(fl.landmark)
                    last_pred = predict_stroke_risk(last_feat, model, scaler)
                else:
                    last_feat, last_pred = None, None
            else:
                now = time.time(); fps = 1/(now-prev+1e-6); prev = now

            # ── Draw cached result on every frame ──
            if last_feat and last_pred:
                frame = draw_overlay(frame, last_feat, last_pred, fps)
            else:
                cv2.putText(frame, 'No face detected', (50,60),
                            cv2.FONT_HERSHEY_SIMPLEX, 1.0, C['orange'], 2)

            if GUI:
                cv2.imshow('Stroke Detection', frame)
                if cv2.getWindowProperty('Stroke Detection', cv2.WND_PROP_VISIBLE) < 1: break
                k = cv2.waitKey(1) & 0xFF
                if k in (ord('q'),ord('Q')): break
                elif k in (ord('s'),ord('S')):
                    shots += 1; fn = f'shot_{shots:02d}.png'
                    cv2.imwrite(fn, frame); print(f"📸 {fn}")
            else:
                clear_output(wait=True)
                # ── V_08: resize preview for faster inline rendering ──
                preview = cv2.resize(frame, (640, 360))
                plt.figure(figsize=(10,5.6)); plt.imshow(cv2.cvtColor(preview,cv2.COLOR_BGR2RGB))
                plt.axis('off'); plt.title(f'Frame {frames+1}/{MAX_FRAMES}  — interrupt to stop')
                plt.show()
                frames += 1
                if frames >= MAX_FRAMES: break
    except KeyboardInterrupt:
        print("\n⏹️  Stopped.")
    finally:
        cap.release()
        if GUI: cv2.destroyAllWindows()
        face_mesh.close()
        print("✅ Camera session ended.")


## Section 7 — Quick Validation with Sample Cases

Verify the prediction pipeline with three hand-crafted cases  
and 10 random rows from the synthetic dataset.


In [ ]:
model  = joblib.load(MODEL_PATH)
scaler = joblib.load(SCALER_PATH)

cases = {
    'Normal (symmetric)':        dict(eye_asymmetry=0.03,mouth_corner_drop=0.02,
                                       eyebrow_height_diff=0.02,nasolabial_asymmetry=0.03,
                                       face_midline_deviation=0.01,mouth_width_asymmetry=0.03),
    'Borderline (mild asym)':    dict(eye_asymmetry=0.13,mouth_corner_drop=0.11,
                                       eyebrow_height_diff=0.09,nasolabial_asymmetry=0.12,
                                       face_midline_deviation=0.06,mouth_width_asymmetry=0.11),
    'High Risk (severe asym)':   dict(eye_asymmetry=0.38,mouth_corner_drop=0.42,
                                       eyebrow_height_diff=0.31,nasolabial_asymmetry=0.37,
                                       face_midline_deviation=0.14,mouth_width_asymmetry=0.33),
}

W = 68
print("="*W)
print(f"{'Case':<35} | {'Prob':>7} | Risk Level")
print("="*W)
for cname, feats in cases.items():
    r = predict_stroke_risk(feats, model, scaler)
    print(f"{cname:<35} | {r['percentage']:>5.1f}%  | {r['risk_level']}")
    # ── V_08: per-feature flag table ──
    print(f"   {'Feature':<28} {'Value':>7}  {'Thresh':>7}  Status")
    print(f"   {'-'*55}")
    for k in FEATURE_COLS:
        v   = feats[k]
        thr = THRESHOLDS[k]
        flag = '✗ HIGH  ' if v > thr else '✓ Normal'
        print(f"   {k:<28} {v:>7.3f}  {thr:>7.2f}  {flag}")
    print()
print("="*W)

# ── 10 random synthetic rows ──
print(f"\n{'#':<3} {'True':<12} {'Pred':<12} {'Prob':>7} | Match")
print("-"*W)
rng = np.random.RandomState(123)
sample = df.loc[rng.choice(df.index, 10, replace=False)]
correct = 0
for i, (_, row) in enumerate(sample.iterrows(), 1):
    feats = {f: row[f] for f in FEATURE_COLS}
    r = predict_stroke_risk(feats, model, scaler)
    true_l = int(row['label']); pred_l = r['label']
    ok = true_l == pred_l; correct += int(ok)
    print(f"{i:<3} {'Stroke' if true_l else 'Normal':<12} {'Stroke' if pred_l else 'Normal':<12} "
          f"{r['percentage']:>5.1f}%  | {'✅' if ok else '❌'}")
print("-"*W)
print(f"Accuracy: {correct}/10 ({correct*10}%)")


## Section 8 — Static Image Analysis

Upload or select a face image.  The system will:
1. Detect face landmarks with MediaPipe  
2. Extract 6 asymmetry features  
3. Predict stroke risk  
4. Show annotated image (with overall asymmetry badge top-right) and gauge chart

> All outputs are **auto-saved** — no prompts.


In [ ]:
# ── Cell 1: Load image & run prediction ──────────────────────
model  = joblib.load(MODEL_PATH)
scaler = joblib.load(SCALER_PATH)

mp_fm     = mp.solutions.face_mesh
face_mesh = mp_fm.FaceMesh(static_image_mode=True,max_num_faces=1,
                             refine_landmarks=True,min_detection_confidence=0.5)

def get_image_path():
    try:
        from google.colab import files
        uploaded = files.upload()
        if uploaded: return list(uploaded.keys())[0]
    except ImportError:
        pass
    found = sorted(sum([glob.glob(e) for e in IMG_EXTS],[]))
    if found:
        print("Images found:")
        for i,f in enumerate(found,1): print(f"  [{i}] {f}")
        print("  [0] Enter path manually")
        c = input(f"Select (0-{len(found)}): ").strip()
        if c.isdigit() and 1<=int(c)<=len(found): return found[int(c)-1]
    return input("Enter image path: ").strip()

IMAGE_PATH = get_image_path()
if not os.path.exists(IMAGE_PATH):
    raise FileNotFoundError(f"Not found: {IMAGE_PATH}")

img_bgr = cv2.imread(IMAGE_PATH)
img_rgb = cv2.cvtColor(img_bgr,cv2.COLOR_BGR2RGB)
res     = face_mesh.process(img_rgb)

if not res.multi_face_landmarks:
    raise ValueError("❌ No face detected. Use a clear front-facing photo.")

fl    = res.multi_face_landmarks[0]
FEATURES = extract_features(fl.landmark)
PREDICTION  = predict_stroke_risk(FEATURES,model,scaler)

IMG_BGR=img_bgr.copy(); IMG_RGB=img_rgb.copy(); FACE_LANDMARKS=fl

pct  = PREDICTION['percentage']
lvl  = PREDICTION['risk_level']
print(f"{'='*52}")
print(f"  STROKE RISK: {pct:.1f}%")
print(f"  Risk Level : {lvl}")
print(f"{'='*52}")
print(f"{'Feature':<28} {'Value':>7}  Status")
print("-"*52)
for k,lbl in {'eye_asymmetry':'Eye Asymmetry','mouth_corner_drop':'Mouth Corner Drop',
               'eyebrow_height_diff':'Eyebrow Height Diff','nasolabial_asymmetry':'Nasolabial Asym',
               'face_midline_deviation':'Face Midline Dev','mouth_width_asymmetry':'Mouth Width Asym'}.items():
    v=FEATURES[k]; s='✗ HIGH' if v>THRESHOLDS[k] else '✓ Normal'
    print(f"  {lbl:<26} {v:>7.3f}  {s}")
print("="*52)
print("\n✅ Run next cell for annotated image.")


In [ ]:
# ─────────────────────────────────────────────
# Section 9 — Cell 2: Annotated Image
# Visual explanation of prediction result
# Research basis:
#   Aldridge et al. (2022)   — explainability; users need to see WHY
#   Gomes et al. (2024)      — lower facial region most discriminative
#   Taufique & Savakis (2021)— landmark-based asymmetry regions
# ─────────────────────────────────────────────

def draw_annotated_image(img_rgb, face_landmarks, features, prediction, thresholds):
    """
    Draw face mesh + colored landmark highlights on the image.
    - Full face mesh: gray (background context)
    - Normal features: green circle
    - HIGH features:   red circle + label
    - Bottom banner:   risk result
    """
    import mediapipe as mp
    if not hasattr(mp, 'solutions'):
        raise RuntimeError(
            "mp.solutions is missing - your mediapipe installation is broken.\n"
            "Go to Section 0 (top of notebook), run the setup cell, "
            "RESTART THE KERNEL, then run this cell again."
        )
    mp_drawing = mp.solutions.drawing_utils
    mp_face_mesh = mp.solutions.face_mesh
    mp_styles = mp.solutions.drawing_styles

    img = img_rgb.copy()
    h, w = img.shape[:2]
    lm = face_landmarks.landmark

    # ── Draw full face mesh (gray, thin) ──
    img_bgr = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    mp_drawing.draw_landmarks(
        image=img_bgr,
        landmark_list=face_landmarks,
        connections=mp_face_mesh.FACEMESH_TESSELATION,
        landmark_drawing_spec=None,
        connection_drawing_spec=mp_drawing.DrawingSpec(
            color=(180, 180, 180), thickness=1, circle_radius=0)
    )
    img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # ── Landmark groups for each feature ──
    # (index in MediaPipe 468-point model)
    feature_landmarks = {
        'eye_asymmetry':          [159, 145, 386, 374],
        'mouth_corner_drop':      [61, 291],
        'eyebrow_height_diff':    [70, 300],
        'nasolabial_asymmetry':   [92, 322, 1],
        'face_midline_deviation': [1, 234, 454],
        'mouth_width_asymmetry':  [61, 291, 13, 14],
    }

    feature_display_names = {
        'eye_asymmetry':          'Eye Asym',
        'mouth_corner_drop':      'Mouth Drop',
        'eyebrow_height_diff':    'Brow Diff',
        'nasolabial_asymmetry':   'Nasolabial',
        'face_midline_deviation': 'Midline',
        'mouth_width_asymmetry':  'Mouth Width',
    }

    # ── Draw colored circles on landmark points ──
    for feat_key, lm_indices in feature_landmarks.items():
        val    = features[feat_key]
        thresh = thresholds[feat_key]
        is_high = val > thresh

        color  = (220, 50, 50)  if is_high else (50, 200, 80)   # red / green
        radius = 8              if is_high else 5

        for idx in lm_indices:
            px = int(lm[idx].x * w)
            py = int(lm[idx].y * h)
            cv2.circle(img, (px, py), radius, color, -1)
            cv2.circle(img, (px, py), radius + 2, (255, 255, 255), 1)  # white border

        # ── Label on first landmark point (HIGH features only) ──
        if is_high:
            px0 = int(lm[lm_indices[0]].x * w)
            py0 = int(lm[lm_indices[0]].y * h)
            label_txt = f"{feature_display_names[feat_key]} ✗"
            cv2.putText(img, label_txt,
                        (px0 + 10, py0 - 5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.45,
                        (220, 50, 50), 1, cv2.LINE_AA)

    # ── Bottom risk banner ──
    banner_h = 60
    overlay  = img.copy()
    cv2.rectangle(overlay, (0, h - banner_h), (w, h), (20, 20, 20), -1)
    img = cv2.addWeighted(overlay, 0.75, img, 0.25, 0)

    risk_pct = prediction['percentage']
    risk_lvl = prediction['risk_level'].split('—')[0].strip()

    color_hint = prediction['color_hint']
    banner_color = {
        'green':  (50, 200, 80),
        'orange': (255, 165, 0),
        'red':    (220, 50, 50),
    }[color_hint]

    cv2.putText(img,
                f"{risk_lvl}  —  {risk_pct:.1f}%",
                (12, h - banner_h + 38),
                cv2.FONT_HERSHEY_DUPLEX, 0.75,
                banner_color, 1, cv2.LINE_AA)

    return img


# ── Generate annotated image ──
annotated = draw_annotated_image(
    IMG_RGB, FACE_LANDMARKS, FEATURES, PREDICTION, THRESHOLDS
)

# ── Display in notebook ──
plt.figure(figsize=(10, 8))
plt.imshow(annotated)
plt.axis('off')
plt.title(
    f"Visual Explanation — Stroke Risk: {PREDICTION['percentage']:.1f}%\n"
    f"🟢 Green = Normal   🔴 Red = Threshold Exceeded",
    fontsize=13, pad=12
)
plt.tight_layout()
plt.show()

# ── Save option ──
save_choice = 'y'  # default to 'yes' for saving
if save_choice == 'y':
    ANNOTATED_PATH = 'stroke_annotated.png'
    plt.figure(figsize=(10, 8))
    plt.imshow(annotated)
    plt.axis('off')
    plt.title(
        f"Visual Explanation — Stroke Risk: {PREDICTION['percentage']:.1f}%\n"
        f"Green = Normal   Red = Threshold Exceeded",
        fontsize=13, pad=12
    )
    plt.tight_layout()
    plt.savefig(ANNOTATED_PATH, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"✅ Saved → {ANNOTATED_PATH}")
else:
    print("⏭️  Skipped — annotated image not saved.")

print("\n✅ Cell 2 done — run Cell 3 for Radar Chart.")


In [ ]:
# ── Cell 3: Gauge Chart ──────────────────────────────────────
def draw_gauge(risk_prob, ax):
    ax.set_xlim(-1.35,1.35); ax.set_ylim(-0.22,1.35)
    ax.set_aspect('equal'); ax.axis('off')

    # Arc zones
    for sd,ed,col,lbl in [(180,135,'#27ae60','LOW\n< 25%'),
                           (135,81,'#f39c12','MODERATE\n25-55%'),
                           (81,0,'#e74c3c','HIGH\n> 55%')]:
        th=np.linspace(np.radians(sd),np.radians(ed),120)
        ro,ri=1.0,0.60
        xs=np.concatenate([ro*np.cos(th),ri*np.cos(th[::-1])])
        ys=np.concatenate([ro*np.sin(th),ri*np.sin(th[::-1])])
        ax.fill(xs,ys,color=col,alpha=0.88,zorder=2)
        mt=np.radians((sd+ed)/2)
        ax.text(0.8*np.cos(mt),0.8*np.sin(mt),lbl,ha='center',va='center',
                fontsize=8.5,color='white',fontweight='bold',zorder=4,multialignment='center')

    # Tick marks
    for ang_pct in np.linspace(0,100,11):
        ang=np.radians(180-ang_pct*1.8)
        ax.plot([0.62*np.cos(ang),1.02*np.cos(ang)],[0.62*np.sin(ang),1.02*np.sin(ang)],
                'w-',lw=1.2,alpha=0.4,zorder=3)
        if ang_pct%25==0:
            ax.text(1.15*np.cos(ang),1.15*np.sin(ang),f'{ang_pct:.0f}%',
                    ha='center',va='center',fontsize=7.5,color='#bbb')

    # Needle with shadow
    na=np.radians(180-risk_prob*180); nl=0.85
    nx,ny=nl*np.cos(na),nl*np.sin(na)
    ax.plot([0,nx+0.01],[0,ny-0.01],color='#333',lw=4,zorder=5,solid_capstyle='round')
    ax.annotate('',xy=(nx,ny),xytext=(0,0),
                arrowprops=dict(arrowstyle='->',color='white',lw=3.2,mutation_scale=20),zorder=6)
    ax.add_patch(plt.Circle((0,0),0.075,color='white',zorder=7))
    ax.add_patch(plt.Circle((0,0),0.045,color='#1a1a2e',zorder=8))

    # Percentage readout
    tcol={'green':'#2ecc71','orange':'#f39c12','red':'#e74c3c'}[PREDICTION['color_hint']]
    ax.text(0,-0.05,f'{risk_prob*100:.1f}%',ha='center',va='top',fontsize=30,
            fontweight='bold',color=tcol,zorder=9)
    ax.text(0,-0.14,'Overall Asymmetry Score',ha='center',va='top',fontsize=9,
            color='#aaa',zorder=9)

fig,ax=plt.subplots(figsize=(9,5.8)); fig.patch.set_facecolor('#0f0f1a'); ax.set_facecolor('#0f0f1a')
draw_gauge(PREDICTION['probability'],ax)
fig.suptitle(f"Asymmetry Score Gauge  |  {PREDICTION['risk_level'].split('—')[0].strip()}\n"
             f"Thresholds: Chen et al. (2022) FAST/BEFAST",
             color='white',fontsize=12)
plt.tight_layout(); plt.show()

# ── Auto-save gauge chart ──
GAUGE_PATH = os.path.join(NOTEBOOK_DIR, 'stroke_gauge.png')
fig2,ax2=plt.subplots(figsize=(9,5.8)); fig2.patch.set_facecolor('#0f0f1a'); ax2.set_facecolor('#0f0f1a')
draw_gauge(PREDICTION['probability'],ax2)
fig2.suptitle(f"Asymmetry Score Gauge  |  {PREDICTION['risk_level'].split('—')[0].strip()}",
              color='white',fontsize=12)
plt.tight_layout()
plt.savefig(GAUGE_PATH,dpi=150,bbox_inches='tight',facecolor='#0f0f1a')
plt.close()
print(f"✅ Gauge chart auto-saved → {GAUGE_PATH}")
print("\n✅ Section 8 complete.")


## ⚠️ Important Disclaimer

> **This system is a SCREENING TOOL, not a diagnostic tool.**
>
> - Flags *potential* stroke risk from facial asymmetry only.
> - Does NOT replace clinical diagnosis by a licensed medical professional.
> - A positive result must prompt immediate medical consultation.
> - False positives and false negatives are inherent to any screening tool.
>
> *"If you notice facial drooping, seek emergency medical help immediately."*  
> — American Stroke Association (FAST protocol)

---

### Project Summary

| Item | Detail |
|---|---|
| Method | MediaPipe Face Mesh + ML Classifier |
| Features | 6 facial asymmetry metrics |
| Training data | Clinically-inspired synthetic (N=3 000) |
| Validation data | Annotated stroke/non-stroke images (100+100) |
| Clinical basis | FAST Protocol, Sunnybrook FGS, CPSS |
| Key references | Herpich 2022, Taufique 2021, Oliveira 2023 |
| Outputs | `stroke_model.pkl`, `stroke_scaler.pkl` |


## Section 10 — Stroke Detection PDF Report Generation

Generate a professional, clinical-grade 2-page PDF scan report using the helper module `generate_report.py`.
The report includes visual explanation (annotated face mesh), asymmetry calibration (gauge chart), clinical thresholds references, input validation, and verification QR code.
It automatically saves to your cross-platform Home Documents folder (`~/Documents/NeuroScan-Report-YYYY-MM-DD.pdf`).

In [ ]:
import os
from generate_report import generate_pdf_report

# Paths of the images generated in earlier sections
annotated_path = os.path.join(NOTEBOOK_DIR, "stroke_annotated.png")
gauge_path = os.path.join(NOTEBOOK_DIR, "stroke_gauge.png")

# Generate the clinical 2-page PDF report (supports A4 or letter page sizes)
success = generate_pdf_report(
    features=FEATURES,
    prediction=PREDICTION,
    annotated_img_path=annotated_path,
    gauge_img_path=gauge_path,
    pagesize_name="letter"  # Options: "letter" or "A4"
)

if success:
    print("\n🏆 Face Scan Report generated successfully in your Documents folder!")
else:
    print("\n❌ Report generation failed. See log output above.")